In [2]:
# Import packages
import PKA_Sleep as PKA
from PKA_Sleep import Graphing_Utils as graph
import numpy as np
import matplotlib.pyplot as plt
graph.make_bigandbold()
from joblib import Parallel, delayed
from scipy.optimize import minimize
import os
import pandas as pd
from scipy.signal import savgol_filter

In [4]:
epoch_len = 4
filter_bounds = [None, None]
binned = False
shuffle_window = 200
experimental_sensor = 'FLIM-mAKAR'
sleep_states = True
microarousals = True
seperate_acqs = False
emp_lifetime = False
gather_timestamps = False
parent_data_directory = '/Volumes/yaochen/Active/Lizzie/FLP_data/'
baseline_only = True
experiment_names = ['ltFLiPAKARmemEEGEMG0072']
mouse_names = ['4610']

raw_datadirs = [os.path.join(parent_data_directory, e) for e in experiment_names]
df = pd.read_excel('/Users/lizzie/Library/CloudStorage/Box-Box/ChenLab/Lizzie/FLiP_Experiment_Summary.xlsx')
baseline_idxs = [(int(d['Baseline Start'].values),int(d['Baseline End'].values)) 
                  for d in [df.loc[df['Experiment Name'] == e] for e in experiment_names]]
excluded_acqs = PKA.choose_excluded_acqs(raw_datadirs, first_acqs = 3, specific_acqs = False, 
                                              pull_baseline = True, baseline_idxs = baseline_idxs)
   
FLP_classes_dicts = PKA.build_classes(experiment_names, mouse_names, epoch_len = epoch_len, 
                                     filter_bounds = filter_bounds, binned = binned, 
                                     shuffle_window = shuffle_window, experimental_sensor = experimental_sensor, 
                                     sleep_states = sleep_states, microarousals = microarousals, 
                                     seperate_acqs = seperate_acqs, emp_lifetime = False,
                                     parent_data_directory = parent_data_directory, gather_timestamps = True, 
                                      exclude_acqs = excluded_acqs)

/var/folders/30/t1sfg_lj50j43rrvkwr7k7mw0000gn/T/ipykernel_14928/4229175756.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  baseline_idxs = [(int(d['Baseline Start'].values),int(d['Baseline End'].values))


You are excluding the first 3 acquisitions of this experiment.
You didn't put any filter values


In [5]:
def fit_animal_bic_fast(
    P2,
    # bounds (use your current ones)
    B1_bounds=(5.0, 300.0),   # decay tau (s)
    B2_bounds=(5.0, 100.0),   # rise tau (s)
    S1_bounds=(0.5, 0.9),     # asymptote for decay
    S2_bounds=(0.5, 0.9),     # asymptote for rise

    # search budget
    n_coarse=2000,            # random samples in coarse pass
    topK=40,                  # keep best K from coarse
    n_refine_local=8,         # run local optimizer from best N seeds

    # coarse eval speed-ups
    subsample_frac=0.25,      # evaluate ~25% of timepoints in coarse pass
    rng_seed=0,
    n_jobs=-1,                # parallel jobs
    verbose=True
):
    rng = np.random.default_rng(rng_seed)

    P2 = np.asarray(P2, float)
    m = np.isfinite(P2)
    idx_all = np.flatnonzero(m)
    if idx_all.size < 20:
        raise ValueError("Too few valid samples for fitting.")

    # -------- Stage A: COARSE random search on subsample --------
    if subsample_frac < 1.0:
        k = max(200, int(np.ceil(subsample_frac * idx_all.size)))
        idx_sub = np.sort(rng.choice(idx_all, size=k, replace=False))
    else:
        idx_sub = idx_all

    logB1_lo, logB1_hi = np.log(B1_bounds[0]), np.log(B1_bounds[1])
    logB2_lo, logB2_hi = np.log(B2_bounds[0]), np.log(B2_bounds[1])

    def draw_candidate():
        B1 = np.exp(rng.uniform(logB1_lo, logB1_hi))
        B2 = np.exp(rng.uniform(logB2_lo, logB2_hi))
        S1 = rng.uniform(*S1_bounds)
        S2 = rng.uniform(*S2_bounds)
        return B1, B2, S1, S2

    def eval_candidate_sub(cand):
        B1, B2, S1, S2 = cand
        # fitS, fitState, err_t, BIC_ar = two_model_fit(P2,40, 1-1/B1, S1, 1-1/B2, S2,0)
        fitS, fitState, err_t, BIC_ar = two_model_fit_lin(P2, 40, B1, B2, plotflag=False, ax=None)
        return BIC_ar

    cands = [draw_candidate() for _ in range(n_coarse)]
    bic_sub = Parallel(n_jobs=n_jobs, prefer="threads")(delayed(eval_candidate_sub)(c) for c in cands)
    order = np.argsort(bic_sub)
    keep_idx = order[:topK]
    kept = [cands[i] for i in keep_idx]

    if verbose:
        best_sub = float(np.min(np.array(bic_sub)[keep_idx]))
        print(f"[coarse] evaluated={n_coarse}, kept={topK}, best BIC (subsample)={best_sub:.3f}")

    # -------- Stage B: refine topK on FULL data --------
    def eval_candidate_full(cand):
        B1, B2, S1, S2 = cand
        # fitS, fitState, err_t, BIC_ar = two_model_fit(P2,40, 1-1/B1, S1, 1-1/B2, S2,0)
        fitS, fitState, err_t, BIC_ar = two_model_fit_lin(P2, 40, B1, B2, plotflag=False, ax=None)
        return BIC_ar

    bic_full = Parallel(n_jobs=n_jobs, prefer="threads")(delayed(eval_candidate_full)(c) for c in kept)
    order2 = np.argsort(bic_full)
    seeds = [kept[i] for i in order2[:n_refine_local]]

    if verbose:
        print(f"[refine] topK→local seeds={n_refine_local}, best BIC (full)={float(np.min(np.array(bic_full)[order2[:1]])):.3f}")

    # -------- Stage C: local optimization on FULL data --------
    def obj(x):
        # x = [logB1, logB2, S1, S2]
        logB1, logB2, S1, S2 = x
        B1 = np.clip(np.exp(logB1), *B1_bounds)
        B2 = np.clip(np.exp(logB2), *B2_bounds)
        S1 = np.clip(S1, *S1_bounds)
        S2 = np.clip(S2, *S2_bounds)
        # fitS, fitState, err_t, BIC_ar = two_model_fit(P2,40, 1-1/B1, S1, 1-1/B2, S2,0)
        fitS, fitState, err_t, BIC_ar = two_model_fit_lin(P2, 40, B1, B2, plotflag=False, ax=None)

        return BIC_ar

    bounds_local = [
        (np.log(B1_bounds[0]), np.log(B1_bounds[1])),
        (np.log(B2_bounds[0]), np.log(B2_bounds[1])),
        (S1_bounds[0], S1_bounds[1]),
        (S2_bounds[0], S2_bounds[1]),
    ]

    best = {"B1": None, "B2": None, "S1": None, "S2": None, "bic": np.inf}

    for s in seeds:
        x0 = np.array([np.log(s[0]), np.log(s[1]), s[2], s[3]], float)
        res = minimize(obj, x0, method="L-BFGS-B", bounds=bounds_local, options=dict(maxiter=200))
        x = res.x
        B1 = float(np.clip(np.exp(x[0]), *B1_bounds))
        B2 = float(np.clip(np.exp(x[1]), *B2_bounds))
        S1 = float(np.clip(x[2], *S1_bounds))
        S2 = float(np.clip(x[3], *S2_bounds))
        # fitS, fitState, err_t, BIC_ar = two_model_fit(P2,40, 1-1/B1, S1, 1-1/B2, S2,0)
        fitS, fitState, err_t, BIC_ar = two_model_fit_lin(P2, 40, B1, B2, plotflag=False, ax=None)

        bic = BIC_ar
        if bic < best["bic"]:
            best.update(dict(B1=B1, B2=B2, S1=S1, S2=S2, bic=float(bic)))

    if verbose:
        print(f"[done] Best: {best}")

    return best




def plot_bic_profiles_1d(
    P2, best, bounds,
    n=80,
    log_params=("B1","B2"),
    rel=True,                 # show ΔBIC w.r.t. min per curve
    figsize=(10,6),
    title="BIC profiles (1D; others fixed)"
):
    """
    Vary each parameter across its bound range while holding the other three
    at 'best'. For each grid point, compute BIC_ar via your two_model_fit.
    """
    names = ["B1","B2","S1","S2"]

    def _eval(B1, B2, S1, S2):
        # fitS, fitState, err_t, BIC_ar = two_model_fit(P2, 40, 1-1/B1, S1, 1-1/B2, S2, 0)
        fitS, fitState, err_t, BIC_ar = two_model_fit_lin(P2, 40, B1, B2, plotflag=False, ax=None)

        return float(BIC_ar)

    grids, curves = {}, {}
    for p in names:
        lo, hi = bounds[p]
        if p in log_params:
            grids[p] = np.exp(np.linspace(np.log(lo), np.log(hi), n))
        else:
            grids[p] = np.linspace(lo, hi, n)

        y = []
        for v in grids[p]:
            pars = dict(best); pars[p] = float(v)
            y.append(_eval(pars["B1"], pars["B2"], pars["S1"], pars["S2"]))
        y = np.array(y, float)
        if rel: y = y - np.nanmin(y)
        curves[p] = y

    fig, axes = plt.subplots(2,2, figsize=figsize, sharey=True)
    axes = axes.ravel()
    for ax, p in zip(axes, names):
        x = grids[p]; y = curves[p]
        ax.plot(x, y, lw=2)
        ax.axvline(best[p], color="k", ls="--", lw=1)
        ax.set_xscale("log" if p in log_params else "linear")
        ax.set_xlabel(p)
        ax.set_ylabel("ΔBIC" if rel else "BIC")
        ax.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

    return {"grid": grids, "bic": curves}

In [16]:
FLP_exp = FLP_classes_dicts['Experiment Classes'][0]
clipped_time_idx = PKA.clip_wake(FLP_exp.SleepStates, slide = 1, thresh = 0.2, max_length = 7200)
tesBest = fit_animal_bic_fast(savgol_filter(FLP_exp.Lifetime[clipped_time_idx[0]:clipped_time_idx[-1]], 11, 2))
bounds = {"B1": (5.0, 300.0), "B2": (5.0, 100.0), "S1": (0.5, 0.9), "S2": (0.5, 0.9)}
out_1d = plot_bic_profiles_1d(savgol_filter(FLP_exp.Lifetime[clipped_time_idx[0]:clipped_time_idx[-1]], 11, 2), tesBest, bounds, n=80)

[coarse] evaluated=2000, kept=40, best BIC (subsample)=-52079.947
[refine] topK→local seeds=8, best BIC (full)=-52079.947
[done] Best: {'B1': 210.44207778595043, 'B2': 30.39608472488032, 'S1': 0.6410189421611837, 'S2': 0.7797111413465855, 'bic': -52632.28174853845}


In [1]:
s1 = -0.0003
s2 = 0.002
fitSL, fitErrorL, fitStateL, BIC_lin = PKA.two_model_fit_lin(
    savgol_filter(FLP_exp.Lifetime[clipped_time_idx[0]:clipped_time_idx[-1]], 11, 2), 40, s1, s2, plotflag=1, ax=None)

NameError: name 'PKA' is not defined